In [1]:
!nvidia-smi

Tue May 12 06:38:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install accelerate
!pip install evaluate
!pip install -U transformers

Found existing installation: transformers 5.8.0
Uninstalling transformers-5.8.0:
  Successfully uninstalled transformers-5.8.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires transformers, which is not installed.
  Using cached transformers-5.8.0-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.8.0-py3-none-any.whl (10.6 MB)


In [3]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
from datasets import load_dataset
import pandas as pd
from datasets import load_dataset

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch
import evaluate

metric = evaluate.load("accuracy")

nltk.download("punkt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [6]:
model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

In [7]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
dataset_news = load_dataset("vietgpt/news_summarization_vi")

Generating train split:   0%|          | 0/65361 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [12]:
dataset_news

DatasetDict({
    train: Dataset({
        features: ['content', 'summary'],
        num_rows: 65361
    })
    test: Dataset({
        features: ['content', 'summary'],
        num_rows: 1000
    })
})

In [13]:
dataset_news["train"]["content"][1]

'Suốt bao năm, để dòng tranh này không bị rơi vào quên lãng, mỗi ngày người ta đều thấy ông Đạt cặm cụi làm nên những bức tranh từ mũi dao, cán đục. Ông bảo, tranh sơn khắc ở nước ta ra đời sớm nhất và còn đẹp hơn cả tranh sơn khắc của Nhật. Quý giá như vậy nên ông chẳng thể để nghề mai một trong sự chông chênh của thời cuộc.\nMột trong những sản phẩm sơn khắc của ông Đạt được trả 25 triệu.\nTheo ông Đạt, thời điểm năm 1945 đến 1995 là lúc tranh sơn khắc ở nước ta phát triển mạnh nhất. Thời điểm đó, các sản phẩm của Hạ Thái chiếm tới 70% hàng xuất khẩu, giải quyết được công ăn việc làm cho người dân trong làng và cả các địa phương khác, đem lại cuộc sống khấm khá cho nhiều hộ gia đình.\nSay mê hội họa từ nhỏ, nên chuyện ông Đạt đến với tranh sơn khắc như một mối duyên tiền định. Khi mới tiếp xúc với những bức tranh này, ông Đạt như bị lôi cuốn chẳng thể nào dứt ra được. Học hết cấp 3, tôi thi vào Đại học sư phạm nhưng sức khỏe không đảm bảo nên xin vào làm thợ vẽ trong xưởng của hợp tá

In [14]:
dataset_news["train"]["summary"][1]

'Ông Đạt Trần Thành là một trong những nghệ nhân sơn khắc của làng nghề Hạ Thái, Hà Nội. Từ năm 1945 đến 1995, ông Đạt đã nỗ lực bảo vệ dòng tranh sơn khắc của nước ta không bị rơi vào quên lãng. Ông Đạt cũng là người đã giới thiệu tranh sơn khắc của nước ta đến với nhiều quốc gia khác. Tuy nhiên, trong giai đoạn khủng hoảng kinh tế Đông Âu từ 1984 đến 1990, làng nghề Hạ Thái đã bước vào thời kỳ suy thoái. Ông Đạt và nhiều người thợ khác đã phải quay về làm ruộng. Tuy nhiên, ông Đạt vẫn nỗ lực bảo vệ dòng tranh sơn khắc của nước ta. Hiện nay, ông Đạt đã truyền cảm hứng và kỹ năng sơn khắc cho các thành viên trong gia đình.'

In [17]:
split_lengths = [len(dataset_news[split])for split in dataset_news]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_news['train'].column_names}")
print("\nContent:")

print(dataset_news["test"][1]["content"])

print("\nSummary:")

print(dataset_news["test"][1]["summary"])

Split lengths: [65361, 1000]
Features: ['content', 'summary']

Content:
Dịp Tết nguyên đán Nhâm Thìn 2012 của ga Sài Gòn, giá vé tàu tết năm nay sẽ tăng 10% giá vé các tàu đối với chiều đông khách, riêng thời gian cao điểm tăng từ 19% đến 39% giá vé. (Ảnh: NLĐ).
Theo ga Sài Gòn, việc điều chỉnh tăng giá vé tàu do từ tết Tân Mão năm 2011 đến nay giá xăng, dầu được liên tục tăng, giãn bớt hành khách ngày cao điểm, khuyến khích hành khách đi lại chiều vắng khách.
Ngoài ra, Đường sắt Việt Nam cũng đồng thời áp dụng việc giảm giá vé cho các chiều vắng khách, cụ thể giảm 79% chiều từ Hà Nội đi Sài Gòn trong giai đoạn trước Tết, giảm 50% chiều từ Sài Gòn đi Hà Nội giai đoạn sau Tết.
Thời gian cao điểm Tết áp dụng đối với tàu số chẵn chạy trước Tết: Từ 0h ngày 14/01/2012 đến hết 24h ngày 21/01/2012 (tức từ ngày 21 tháng Chạp đến ngày 28 tháng Chạp năm Tân Mão).
Đối với tàu số lẻ chạy sau Tết: Từ 0h ngày 26/01/2012 đến hết 24h ngày 03/02/2012 (tức từ ngày 4 tháng Giêng đến hết ngày 12 tháng Giê

In [18]:
def convert_examples_to_features(example_batch):

    model_inputs = tokenizer(
        example_batch["content"],
        max_length=1024,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [19]:
dataset_news_pt = dataset_news.map(convert_examples_to_features, batched = True)

Map:   0%|          | 0/65361 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [20]:
dataset_news_pt["train"]

Dataset({
    features: ['content', 'summary', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 65361
})

In [21]:
dataset_news_pt["train"]["input_ids"][1]

[7410,
 105,
 144,
 110,
 51487,
 3178,
 105,
 208,
 108,
 110,
 105,
 3138,
 105,
 4652,
 59721,
 1467,
 3178,
 105,
 415,
 110,
 19429,
 105,
 4652,
 3027,
 105,
 6114,
 105,
 457,
 2294,
 105,
 554,
 25773,
 105,
 454,
 3834,
 105,
 4652,
 108,
 2653,
 105,
 457,
 110,
 4652,
 105,
 415,
 110,
 4652,
 105,
 457,
 23048,
 110,
 105,
 1858,
 110,
 307,
 105,
 415,
 110,
 105,
 4652,
 110,
 105,
 144,
 2895,
 105,
 208,
 2895,
 105,
 457,
 3834,
 105,
 208,
 3178,
 105,
 454,
 110,
 23774,
 105,
 4652,
 3027,
 105,
 1152,
 59721,
 1467,
 110,
 144,
 105,
 2653,
 105,
 457,
 7447,
 554,
 108,
 2895,
 105,
 454,
 110,
 105,
 1152,
 107,
 110,
 105,
 4652,
 3027,
 105,
 554,
 108,
 59721,
 1467,
 110,
 116,
 105,
 454,
 110,
 19429,
 105,
 1152,
 110,
 105,
 3178,
 105,
 1152,
 23048,
 26997,
 110,
 105,
 457,
 110,
 116,
 105,
 208,
 110,
 23774,
 105,
 144,
 2294,
 105,
 2895,
 105,
 454,
 110,
 105,
 1379,
 5124,
 105,
 454,
 2895,
 105,
 59721,
 1467,
 110,
 116,
 105,
 454,
 110,
 19

In [22]:
dataset_news_pt["train"]["attention_mask"][1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,


In [23]:
dataset_news_pt["train"]["labels"][1]

[110,
 105,
 4652,
 110,
 105,
 144,
 21862,
 105,
 454,
 14085,
 105,
 23774,
 3834,
 105,
 2653,
 105,
 144,
 110,
 144,
 60830,
 110,
 23774,
 105,
 4652,
 3178,
 19836,
 105,
 110,
 23774,
 105,
 454,
 110,
 116,
 105,
 454,
 110,
 19429,
 105,
 1152,
 2895,
 105,
 304,
 3834,
 105,
 4652,
 3178,
 19836,
 105,
 1176,
 105,
 14085,
 105,
 457,
 108,
 1176,
 105,
 1101,
 105,
 457,
 107,
 781,
 105,
 3178,
 105,
 208,
 26402,
 110,
 105,
 454,
 19567,
 110,
 105,
 4652,
 110,
 105,
 144,
 110,
 105,
 3178,
 105,
 3834,
 105,
 1152,
 3027,
 105,
 554,
 2294,
 105,
 3138,
 105,
 4652,
 59721,
 1467,
 110,
 116,
 105,
 454,
 110,
 19429,
 105,
 1152,
 2895,
 105,
 304,
 3178,
 105,
 1152,
 23048,
 110,
 19429,
 105,
 4652,
 3027,
 105,
 6114,
 105,
 457,
 2294,
 105,
 554,
 25773,
 105,
 454,
 3834,
 105,
 4652,
 107,
 1]

In [24]:
# Training

from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [25]:
!pip install -U transformers accelerate

import transformers

print(transformers.__version__)

5.8.0


In [26]:
from transformers import Seq2SeqTrainingArguments

trainer_args = Seq2SeqTrainingArguments(

    output_dir='pegasus-news',

    num_train_epochs=1,

    warmup_steps=500,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    weight_decay=0.01,

    logging_steps=10,

    eval_strategy='steps',
    eval_steps=500,

    save_steps=500,

    gradient_accumulation_steps=16,

    predict_with_generate=True
)

In [29]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model_pegasus,

    args=trainer_args,

    processing_class=tokenizer,

    data_collator=seq2seq_data_collator,

    train_dataset=dataset_news_pt["train"],

    eval_dataset=dataset_news_pt["test"]
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss


In [ ]:
# Evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]



def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="article",
                               column_summary="highlights"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024,  truncation=True,
                        padding="max_length", return_tensors="pt")

        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                         attention_mask=inputs["attention_mask"].to(device),
                         length_penalty=0.8, num_beams=8, max_length=128)
        ''' parameter for length penalty ensures that the model does not generate sequences that are too long. '''

        # Finally, we decode the generated texts,
        # replace the  token, and add the decoded texts with the references to the metric.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                clean_up_tokenization_spaces=True)
               for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]


        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    #  Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score


In [ ]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
rouge_metric = load_metric('rouge')

In [ ]:
score = calculate_metric_on_test_ds(
    dataset_news['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
)

rouge_dict = dict((rn, score[rn].mid.fmeasure ) for rn in rouge_names )

pd.DataFrame(rouge_dict, index = [f'pegasus'] )

In [ ]:
## Save model
model_pegasus.save_pretrained("pegasus-mask-model")

In [ ]:
## Save tokenizer
tokenizer.save_pretrained("tokenizer")

In [ ]:
#Load

tokenizer = AutoTokenizer.from_pretrained("/content/tokenizer")

In [ ]:
#Prediction

gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}



sample_text = dataset_mask["test"][0]["dialogue"]

reference = dataset_mask["test"][0]["summary"]

pipe = pipeline("summarization", model="pegasus-mask-model",tokenizer=tokenizer)

##
print("Dialogue:")
print(sample_text)


print("\nReference Summary:")
print(reference)


print("\nModel Summary:")
print(pipe(sample_text, **gen_kwargs)[0]["summary_text"])